In [ ]:
import os
import sys

sys.path.insert(0, os.path.dirname(os.getcwd()))

In [ ]:
from dataclasses import dataclass
from functools import partial

import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from src.model import (
    WhisperAccentForConditionalGeneration,
    WhisperAccentProcessor,
    register_whisper_accent,
)
from src.model.tokenization import ACCENTS
from src.train.dataset import DataCollatorSpeechSeq2SeqWithPadding, WhisperDataset

register_whisper_accent()

In [ ]:
@dataclass
class LoraArguments:
    lora_enable: bool = True
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    lora_bias: str = "none"
    use_rslora: bool = True
    task_type: str = "SEQ_2_SEQ_LM"


@dataclass
class ModelArguments:
    model_path: str = "openai/whisper-tiny.en"
    is_multilingual: bool = False

In [ ]:
model_args = ModelArguments()
lora_args = LoraArguments()

In [ ]:
def processor_init(model_args, max_length):
    processor = WhisperAccentProcessor.from_pretrained(model_args.model_path)
    processor.tokenizer.add_special_tokens(
        {
            "additional_special_tokens": list(ACCENTS.values()),
            "bos_token": "<|startoftranscript|>",
        }
    )
    processor.tokenizer.model_max_length = max_length
    return processor


processor = processor_init(model_args, 255)

In [ ]:
def model_init(model_args, lora_args, processor, max_length):
    model = WhisperAccentForConditionalGeneration.from_pretrained(model_args.model_path)
    model.resize_token_embeddings(len(processor.tokenizer))
    model.generation_config.accent_to_id = {
        k: v for k, v in processor.tokenizer.vocab.items() if k in ACCENTS.values()
    }
    model.generation_config.max_length = max_length
    model.proj_out = nn.Linear(
        model.proj_out.in_features,
        len(processor.tokenizer),
        bias=model.proj_out.bias is not None,
    )
    model.tie_weights()

    if lora_args.lora_enable:
        target_modules = []
        for name, _ in model.named_modules():
            m_list = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]
            if "model.decoder" in name and any(suffix in name for suffix in m_list):
                target_modules.append(name)

        accent_token_indices = list(model.generation_config.accent_to_id.values())

        lora_config = LoraConfig(
            r=lora_args.lora_r,
            lora_alpha=lora_args.lora_alpha,
            lora_dropout=lora_args.lora_dropout,
            bias=lora_args.lora_bias,
            use_rslora=lora_args.use_rslora,
            target_modules=target_modules,
            trainable_token_indices=accent_token_indices,
            task_type=lora_args.task_type,
            ensure_weight_tying=True,
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()

    if model.generation_config.is_multilingual:
        model.generation_config.language = "en"
        model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = None

    return model

In [ ]:
collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
)

train_dataset = WhisperDataset(
    data_path="westbrook/English_Accent_DataSet",
    processor=processor,
    split="train",
    shuffle=True,
    num_proc=16,
)

eval_dataset = WhisperDataset(
    data_path="westbrook/English_Accent_DataSet",
    processor=processor,
    split="validation",
    shuffle=False,
    num_proc=16,
)

In [ ]:
import datetime

run_name = (
    f"whisper-accent-tiny-en-lora-{datetime.datetime.now().strftime('%Y%m%d-%H%M')}"
)
output_dir = "/workspace/whisper-accent-tiny.en"

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    # max_steps=1000,
    max_steps=500,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=10,
    save_strategy="steps",
    # save_steps=200,
    save_steps=100,
    save_total_limit=100,
    bf16=True,
    fp16=False,
    eval_steps=100,
    run_name=run_name,
    optim="adamw_torch",
    # optim_args={"betas": (0.9, 0.999), "eps": 1e-8, "weight_decay": 0.01},
    report_to=["tensorboard"],
    push_to_hub=True,
    hub_model_id="mavleo96/whisper-accent-tiny.en",
    hub_strategy="all_checkpoints",
    gradient_checkpointing=False,
    predict_with_generate=True,
    remove_unused_columns=False,
)

In [ ]:
from transformers import EvalPrediction

from src.utils.metrics import compute_wer


def compute_metrics(eval_pred: EvalPrediction):
    predictions = eval_pred.predictions
    label_ids = eval_pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    raw_pred_str = processor.tokenizer.batch_decode(
        predictions, skip_special_tokens=True
    )
    pred_str = [processor.tokenizer.normalize(s) for s in raw_pred_str]

    wer, _ = compute_wer(pred_str, label_str)
    return {"wer": wer}

In [ ]:
model_init_fn = partial(
    model_init,
    model_args=model_args,
    lora_args=lora_args,
    processor=processor,
    max_length=255,
)
trainer = Seq2SeqTrainer(
    # model=model,
    model_init=model_init_fn,
    args=training_args,
    data_collator=collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.model.merge_and_unload().save_pretrained(f"{output_dir}")

In [ ]:
processor.save_pretrained(f"{output_dir}")

In [ ]:
trainer.push_to_hub()

In [ ]:
accent_embeddings = model.model.decoder.embed_tokens.weight[
    list(model.generation_config.accent_to_id.values()), :
]
accent_embeddings.shape

In [ ]:
torch.nn.functional.cosine_similarity(accent_embeddings, accent_embeddings, dim=1)